# Rancher Kubernetes (RKE2) on FABRIC via Ansible

This notebook provisions a **2-node** FABRIC slice on a single site and uses Ansible to deploy [RKE2](https://docs.rke2.io/) (Rancher Kubernetes Engine 2).

| Role | Node | Resources |
|------|------|-----------|
| Control plane (`rke2-server`) | `node1` | 8 cores, 16 GB RAM |
| Worker (`rke2-agent`) | `node2` | 8 cores, 16 GB RAM |

Cluster traffic uses a private L2 dataplane (`192.168.1.0/24`). Ansible reaches the nodes over the FABRIC management network.

Ansible inventory and playbooks are generated at runtime from `478_examples/templates/` into `478_examples/playbook/` using `utils/ansible.py`.

## Import the FABlib Library

In [5]:
from ipaddress import IPv4Network
import random
import sys
from pathlib import Path

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "utils" / "ansible.py").exists():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise ImportError("Could not find repo root containing utils/ansible.py")

from utils.ansible import generate_rke2_playbooks

fablib = fablib_manager()
fablib.verify_and_configure()

User: lngo@wcupa.edu bastion key is valid!
Configuration is valid
User: lngo@wcupa.edu bastion key is valid!
Configuration is valid
Please save the config!


In [13]:
manager = fablib.get_manager()

users = manager.list_project_users(
    project_uuid=fablib.get_project_id()
)
for user in users:
    print(user)

{'project_uuid': '8b6dfc51-02ad-4f00-a389-6e75d8b61a26', 'user_uuid': 'f749151c-43ad-425e-9131-5e8ba80188e6', 'email': 'TG996676@wcupa.edu', 'name': 'Tyler Geiger', 'roles': ['member'], 'role': 'member'}
{'project_uuid': '8b6dfc51-02ad-4f00-a389-6e75d8b61a26', 'user_uuid': '336bd639-ebbc-4b63-9de9-b2a8fae86b0d', 'email': 'RF950880@wcupa.edu', 'name': 'Ryan Fioravanti', 'roles': ['member'], 'role': 'member'}
{'project_uuid': '8b6dfc51-02ad-4f00-a389-6e75d8b61a26', 'user_uuid': 'c7d5120b-a3cc-4664-ae7f-05dbcbac4cd0', 'email': 'MP1011698@wcupa.edu', 'name': 'Matthew Potts', 'roles': ['member'], 'role': 'member'}
{'project_uuid': '8b6dfc51-02ad-4f00-a389-6e75d8b61a26', 'user_uuid': '6bde0c39-8162-44a9-b09b-f8b4bef0e4bd', 'email': 'LY996437@wcupa.edu', 'name': 'Logan Yates', 'roles': ['member'], 'role': 'member'}
{'project_uuid': '8b6dfc51-02ad-4f00-a389-6e75d8b61a26', 'user_uuid': 'acc7ba2f-e01d-417a-b8ba-b719acacc138', 'email': 'SC942958@wcupa.edu', 'name': 'Seth Conley', 'roles': ['membe

## Find a Site With Enough Capacity

We need **2 nodes × 8 cores × 10 GB RAM** on one site. Requirements are scaled by `1.2` so a busy site is less likely to fail mid-submit.

In [3]:
resources = fablib.get_resources()
resources.update()

nodesReq = 2
coresReq = 8
ramReq = 10

# Scale up quite a bit to ensure there are plenty of resources available.
totalCoreAvail = nodesReq * coresReq * 5
totalRamAvail = nodesReq * ramReq * 5

usableSite = []
siteList = resources.get_site_names()
for site in siteList:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores >= totalCoreAvail and ram >= totalRamAvail:
        usableSite.append(site)

print(f"Need ~{totalCoreAvail:.0f} cores and ~{totalRamAvail:.0f} GB RAM available")
print(f"Usable sites ({len(usableSite)}): {usableSite}")
assert usableSite, "No site currently has enough free cores/RAM for this slice"

Need ~80 cores and ~100 GB RAM available
Usable sites (23): ['PRIN', 'TACC', 'MASS', 'WASH', 'DALL', 'MAX', 'SALT', 'NCSA', 'HAWI', 'FIU', 'GATECH', 'CLEM', 'CERN', 'NEWY', 'TOKY', 'MICH', 'UTAH', 'SRI', 'RUTG', 'GPN', 'SEAT', 'CIEN', 'EDUKY']


## Create the Slice

Site selection is random among usable sites so concurrent class users are less likely to collide.

In [4]:
#siteName = random.choice(usableSite)
siteName = 'FIU'
sliceName = "RancherK8s-2"
network_name = "rke2net"
print(f"Selected site: {siteName}")

slice = fablib.new_slice(name=sliceName)
net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

for i in range(1, nodesReq + 1):
    node = slice.add_node(
        name=f"node{i}",
        site=siteName,
        cores=coresReq,
        ram=ramReq,
        disk=50,
        image="default_ubuntu_22",
    )
    iface = node.add_component(model="NIC_Basic", name="nic").get_interfaces()[0]
    iface.set_mode("config")
    net.add_interface(iface)

slice.submit()


Retry: 8, Time: 187 sec


ID,1c13c57d-b469-434d-b88e-334fc87b2d99
Name,RancherK8s-2
Lease Expiration (UTC),2026-09-10 16:18:46 +0000
Lease Start (UTC),2026-09-09 16:18:46 +0000
Project ID,8b6dfc51-02ad-4f00-a389-6e75d8b61a26
State,StableOK
Email,lngo@wcupa.edu
UserId,8eecd713-fa8f-4b3b-8883-1ff9b021fa53


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
b5b6d25f-67ba-4e1b-9c18-3bcd967dea5c,node1,8,16,100,default_ubuntu_22,qcow2,fiu-w3.fabric-testbed.net,FIU,ubuntu,131.94.57.62,Active,,ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@131.94.57.62,/home/fabric/.ssh/slice_key.pub,/home/fabric/.ssh/slice_key
de877438-2f20-4734-990f-f12d36b3d5de,node2,8,16,100,default_ubuntu_22,qcow2,fiu-w3.fabric-testbed.net,FIU,ubuntu,131.94.57.32,Active,,ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@131.94.57.32,/home/fabric/.ssh/slice_key.pub,/home/fabric/.ssh/slice_key


ID,Name,Layer,Type,Site,Gateway,Subnet,State,Error
8432ffc0-a963-4a6c-a51e-449b9bc24b95,rke2net,L2,L2Bridge,FIU,None,192.168.1.0/24,Active,


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
node1-nic-p1,p1,node1,rke2net,100,,0A:C1:8C:63:34:03,enp7s0,enp7s0,config,fe80::8c1:8cff:fe63:3403,4,HundredGigE0/0/0/9
node2-nic-p1,p1,node2,rke2net,100,,0A:E7:B1:EA:4D:32,enp7s0,enp7s0,config,fe80::8e7:b1ff:feea:4d32,4,HundredGigE0/0/0/9



Time to print interfaces 188 seconds


'1c13c57d-b469-434d-b88e-334fc87b2d99'

In [5]:
import time

while True:
    time.sleep(10)
    slice.update()
    slice_state = slice.get_state()
    print(f"Slice state: {slice_state}")
    if slice_state == "Closing":
        print(f"Need to find new site")
        break
    else: 
        print("Slice stable:", slice.isStable())
    nodes = slice.get_nodes()
    if all(node.get_management_ip() is not None for node in nodes):
        for node in nodes:
            print("----", node.get_name(), "----")
            print("reservation state:", node.get_reservation_state())
            print("management ip:", node.get_management_ip())
            print("username:", node.get_username())
            print("error:", node.get_error_message())
            print(node.get_ssh_command())
        break

Slice state: StableOK
Slice stable: True
---- node1 ----
reservation state: Active
management ip: 131.94.57.62
username: ubuntu
error: 
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@131.94.57.62
---- node2 ----
reservation state: Active
management ip: 131.94.57.32
username: ubuntu
error: 
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@131.94.57.32


## Configure Dataplane Networking

Assign private IPs on the L2 network. RKE2 will advertise and join using these addresses.

In [6]:
for i in range(1, nodesReq + 1):
    node = slice.get_node(name=f"node{i}")
    iface = node.get_interface(network_name=network_name)
    iface.ip_link_up()
    iface.ip_addr_add(
        addr=f"192.168.1.{i}",
        subnet=IPv4Network("192.168.1.0/24"),
    )
    print(f"{node.get_name()} -> 192.168.1.{i}")

node1 -> 192.168.1.1
node2 -> 192.168.1.2


**Keep re-running the cell below until every node can ping every other node.**

In [7]:
for i in range(1, nodesReq):
    src = slice.get_node(name=f"node{i}")
    for j in range(i + 1, nodesReq + 1):
        des = slice.get_node(name=f"node{j}")
        des_addr = des.get_interface(network_name=network_name).get_ip_addr()
        print(f"{src.get_name()} is pinging {des.get_name()} at {des_addr} ========")
        stdout, stderr = src.execute(f"ping -c 2 {des_addr}")

node1 is pinging node2 at 192.168.1.2 ========
PING 192.168.1.2 (192.168.1.2) 56(84) bytes of data.
64 bytes from 192.168.1.2: icmp_seq=1 ttl=64 time=0.305 ms
64 bytes from 192.168.1.2: icmp_seq=2 ttl=64 time=0.096 ms

--- 192.168.1.2 ping statistics ---
2 packets transmitted, 2 received, 0% packet loss, time 1018ms
rtt min/avg/max/mdev = 0.096/0.200/0.305/0.104 ms


## Generate Ansible files from templates

`utils/ansible.py` fills `__PLACEHOLDER__` values from the live slice and writes:

- `478_examples/playbook/inventory.yml`
- `478_examples/playbook/playbook-prereqs.yml`
- `478_examples/playbook/playbook-rke2.yml`

- `node1` → `rke2_servers` (control plane)
- `node2` → `rke2_agents` (worker)

A shared `rke2_token` is written into inventory so agents can join without scraping the server token file by hand.

In [8]:
inventory = generate_rke2_playbooks(slice, fablib, network_name)
print(inventory)

Wrote playbook/inventory.yml
all:
  vars:
    ansible_become: true
    ansible_ssh_private_key_file: "/home/fabric/.ssh/slice_key"
    ansible_ssh_common_args: "-F /home/fabric/work/fabric_config/ssh_config"
    rke2_token: "fabric-rke2-cluster-token"
    rke2_server_private_ip: "192.168.1.1"

  children:
    rke2_servers:
      hosts:
        node1:
          ansible_host: "131.94.57.62"
          ansible_user: "ubuntu"
          ansible_python_interpreter: "/usr/bin/python3"
          private_ip: "192.168.1.1"

    rke2_agents:
      hosts:
        node2:
          ansible_host: "131.94.57.32"
          ansible_user: "ubuntu"
          ansible_python_interpreter: "/usr/bin/python3"
          private_ip: "192.168.1.2"




## Playbook: Host Prerequisites

Disables swap, loads `overlay` / `br_netfilter`, enables IP forwarding, and writes `/etc/hosts` entries for the cluster nodes.

In [9]:
!ansible-playbook -i ../playbook/inventory.yml ../playbook/playbook-prereqs.yml


PLAY [Prepare hosts for RKE2] **************************************************

TASK [Gathering Facts] *********************************************************
ok: [node2]
ok: [node1]

TASK [Install prerequisite packages] *******************************************
changed: [node2]
changed: [node1]

TASK [Disable swap (required by Kubernetes)] ***********************************
ok: [node1]
ok: [node2]

TASK [Ensure swap stays disabled across reboots] *******************************
ok: [node2]
ok: [node1]

TASK [Load overlay kernel module] **********************************************
ok: [node1]
ok: [node2]

TASK [Load br_netfilter kernel module] *****************************************
ok: [node1]
ok: [node2]

TASK [Persist kernel module loads] *********************************************
changed: [node2]
changed: [node1]

TASK [Configure sysctl for Kubernetes networking] ******************************
changed: [node1]
changed: [node2]

TASK [Apply sysctl settings] **********

## Playbook: Install RKE2

Declarative take on the [RKE2 quick start](https://docs.rke2.io/install/quickstart):

1. Write `config.yaml` so the server advertises on the dataplane IP
2. Install and start `rke2-server` on `node1`
3. Install and start `rke2-agent` on `node2` using the shared token
4. Deploy a small `nginx` Deployment + NodePort Service (`30080`) for a smoke test

First-time install commonly takes **10–15 minutes** while images are pulled.

In [10]:
!ansible-playbook -i ../playbook/inventory.yml ../playbook/playbook-rke2.yml


PLAY [Bootstrap RKE2 server (control plane)] ***********************************

TASK [Gathering Facts] *********************************************************
ok: [node1]

TASK [Ensure RKE2 config directory exists] *************************************
changed: [node1]

TASK [Discover dataplane interface for private IP] *****************************
ok: [node1]

TASK [Show dataplane interface] ************************************************
ok: [node1] => {
    "msg": "Using dataplane iface enp7s0 for 192.168.1.1"
}

TASK [Write RKE2 server config (advertise on private dataplane IP)] ************
changed: [node1]

TASK [Install RKE2 server] *****************************************************
changed: [node1]

TASK [Ensure server manifests directory exists] ********************************
changed: [node1]

TASK [Write Canal HelmChartConfig for FABRIC dataplane] ************************
changed: [node1]

TASK [Enable and start rke2-server] ***************************************

## Verify the Cluster

In [11]:
server = slice.get_node("node1")

print("==== nodes ====")
stdout, stderr = server.execute("kubectl get nodes -o wide", quiet=True)
print(stdout)

print("==== nginx-demo ====")
stdout, stderr = server.execute("kubectl get pods,svc -l app=nginx-demo -o wide", quiet=True)
print(stdout)

print("==== curl via NodePort on dataplane ====")
stdout, stderr = server.execute("curl -s -o /dev/null -w '%{http_code}\n' http://192.168.1.1:30080/", quiet=True)
print("HTTP status:", stdout.strip())

==== nodes ====
NAME    STATUS   ROLES                AGE     VERSION          INTERNAL-IP   EXTERNAL-IP   OS-IMAGE             KERNEL-VERSION               CONTAINER-RUNTIME
node1   Ready    control-plane,etcd   2m10s   v1.36.4+rke2r1   192.168.1.1   <none>        Ubuntu 22.04.5 LTS   5.15.0-185-generic (amd64)   containerd://2.3.4-k3s1.36
node2   Ready    <none>               65s     v1.36.4+rke2r1   192.168.1.2   <none>        Ubuntu 22.04.5 LTS   5.15.0-185-generic (amd64)   containerd://2.3.4-k3s1.36

==== nginx-demo ====
NAME                             READY   STATUS    RESTARTS   AGE   IP           NODE    NOMINATED NODE   READINESS GATES
pod/nginx-demo-6f8d7bb5d-grfw9   1/1     Running   0          8s    10.42.0.13   node1   <none>           <none>
pod/nginx-demo-6f8d7bb5d-sfchw   1/1     Running   0          8s    10.42.1.4    node2   <none>           <none>

==== curl via NodePort on dataplane ====
HTTP status: 200


## Useful Follow-ups

On `node1` (after SSH):

```bash
kubectl get pods -A
kubectl describe node node1
sudo journalctl -u rke2-server -f
```

On `node2`:

```bash
sudo journalctl -u rke2-agent -f
```

To reach the NodePort from your laptop, create an SSH tunnel to `node1:30080` (same pattern as the Docker Swarm notebook).

## Start the SSH Tunnel

- Create SSH Tunnel Configuration `fabric_ssh_tunnel_tools.zip`
- Download your custom `fabric_ssh_tunnel_tools.zip` tarball from the `fabric_config` folder.  
- Untar the tarball and put the resulting folder (`fabric_ssh_tunnel_tools`) somewhere you can access it from the command line.
- Open a terminal window. (Windows: use `powershell`) 
- Use `cd` to navigate to the `fabric_ssh_tunnel_tools` folder.
- Run the following command to setup permission correctly

```bash
chmod 600 slice_key fabric-bastion-key
```

- In your terminal, run the command that results from running the following cell (leave the terminal window open).

In [3]:
fablib.create_ssh_tunnel_config(overwrite=True)


SSH tunnel config created and zipped at: /home/fabric/work/fabric_config/fabric_ssh_tunnel_tools.tgz

Download Instructions:
Download your custom `fabric_ssh_tunnel_tools.tgz` file from the `fabric_config` folder.

Usage Instructions:
1. Unzip the archive and place the resulting `fabric_ssh_tunnel_tools/` folder somewhere accessible from your terminal.
2. Open a terminal window (on Windows, use PowerShell).
3. Use `cd` to navigate into the `fabric_ssh_tunnel_tools` folder.
4. In your terminal, run the SSH tunnel command generated by the next notebook cell.
    


In [28]:
import os
# Port on your local machine that you want to map the web server to. This should be a port that you have specified on 
# docker-compose.yml

server = slice.get_node("node1")
target_host=f'{server.get_username()}@{server.get_management_ip()}'

print(f"Open a terminal tab here and run the following command to connect to the RKE2 server: \n")
print(f'ssh  -i {os.path.basename(fablib.get_default_slice_public_key_file())[:-4]} -F ssh_config {target_host}')
print(f"\nRun the following command once you are connected: \n")
print("kubectl port-forward --address 127.0.0.1 svc/nginx-demo 8080:80")

Open a terminal tab here and run the following command to connect to the RKE2 server: 

ssh  -i slice_key -F ssh_config ubuntu@2001:400:a100:3040:f816:3eff:fe0e:3a14

Run the following command once you are connected: 

kubectl port-forward --address 127.0.0.1 svc/nginx-demo 8080:80


In [29]:
import os
# Port on your local machine that you want to map the web server to. This should be a port that you have specified on 
# docker-compose.yml

server = slice.get_node("node1")
local_port='5555'
# We use 0.0.0.0 because we want the ability to forward this interface outside of the container. 
local_host='0.0.0.0'

# Port on the node used by the web server
target_port='30080'

# Username/node on FABRIC
target_host=f'{server.get_username()}@{server.get_management_ip()}'
print(f"Open another terminal tab here and run the following command to open the SSH tunnel: \n")
print(f'ssh  -L {local_host}:{local_port}:127.0.0.1:{target_port} -i {os.path.basename(fablib.get_default_slice_public_key_file())[:-4]} -F ssh_config {target_host}')

Open another terminal tab here and run the following command to open the SSH tunnel: 

ssh  -L 0.0.0.0:5555:127.0.0.1:30080 -i slice_key -F ssh_config ubuntu@2001:400:a100:3040:f816:3eff:fe0e:3a14


Open a new browser tab and visit `127.0.0.1:5555`

## Cleanup

In [ ]:
# Uncomment when you are finished
slice.delete()